# 02 — Modèle de churn : régression logistique vs XGBoost, explicabilité SHAP

**Objectif :** prédire quels clients vont partir, et surtout **expliquer pourquoi**.

**Plan :** 1. données · 2. comparaison des modèles (validation croisée) · 3. évaluation sur le jeu de test ·
4. explicabilité globale (SHAP) · 5. explication d'un client · 6. le pont avec les avis · 7. sauvegarde · 8. synthèse

Toute la logique est dans `modeling/churn_model.py` (classe `ModeleChurn`, testée dans `tests/test_churn_model.py`) :
le notebook ne fait qu'utiliser et présenter.

In [ ]:
import sys
import warnings
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import shap
from sklearn.metrics import confusion_matrix, precision_recall_curve, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

sys.path.append("..")  # pour importer le code du projet depuis notebooks/
from modeling.churn_model import ModeleChurn, charger_donnees, importance_par_motif, preparer_X_y

warnings.filterwarnings("ignore", category=UserWarning)
BLEU, ORANGE, GRIS = "#2a78d6", "#eb6834", "#52514e"
COULEURS = {"logistique": BLEU, "xgboost": ORANGE}
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})
IMG = Path("../docs/img")
IMG.mkdir(parents=True, exist_ok=True)
DB = Path("../data/telco360.duckdb")

## 1. Les données

In [ ]:
df = charger_donnees(DB)
X, y = preparer_X_y(df)
print(f"{X.shape[0]} clients, {X.shape[1]} variables explicatives, churn = {y.mean():.1%}")

# 80 % pour apprendre, 20 % mis de côté pour l'évaluation finale.
# stratify=y : même proportion de churn (≈ 26,5 %) dans les deux parties.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(f"train : {len(X_train)} clients · test : {len(X_test)} clients")

`preparer_X_y` retire `customer_id` et `montant_total` (redondant : ≈ ancienneté × montant mensuel, vu dans l'EDA),
passe les oui/non en 0/1 et transforme les catégories (contrat, internet, paiement…) en indicatrices.

## 2. Comparaison des modèles (validation croisée)

On compare une **régression logistique** (référence simple et interprétable) et **XGBoost** (plus flexible).
La **validation croisée à 5 plis** sur le jeu d'entraînement mesure la performance et sa **stabilité** :
un score qui varie beaucoup d'un pli à l'autre n'est pas fiable.

Les deux modèles compensent le déséquilibre des classes (`class_weight="balanced"` / `scale_pos_weight`).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultats_cv = {}
for type_modele in ModeleChurn.TYPES:
    estimateur = ModeleChurn(type_modele)._construire(y_train)
    scores = cross_val_score(estimateur, X_train, y_train, cv=cv, scoring="roc_auc")
    resultats_cv[type_modele] = {"ROC-AUC moyen": scores.mean().round(3), "écart-type": scores.std().round(3)}
pd.DataFrame(resultats_cv).T

## 3. Évaluation sur le jeu de test

In [ ]:
modeles = {t: ModeleChurn(t).train(X_train, y_train) for t in ModeleChurn.TYPES}
pd.DataFrame({t: m.evaluer(X_test, y_test) for t, m in modeles.items()}).T

**Lecture des métriques** (seuil de décision 0,5) :
- **ROC-AUC** : capacité à classer un partant au-dessus d'un non-partant (0,5 = hasard, 1 = parfait) ;
- **PR-AUC** : même idée, centrée sur la classe rare (les partants) — plus exigeante ici que la ROC-AUC ;
- **recall** : part des vrais partants détectés ; **precision** : part des alertes qui sont de vrais partants.

Pour une équipe rétention, rater un partant coûte en général plus cher qu'une offre envoyée à tort :
on privilégie donc le **recall**, quitte à baisser la précision.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for type_modele, modele in modeles.items():
    proba = modele.predict_proba(X_test)
    fpr, tpr, _ = roc_curve(y_test, proba)
    prec, rec, _ = precision_recall_curve(y_test, proba)
    axes[0].plot(fpr, tpr, color=COULEURS[type_modele], lw=2, label=type_modele)
    axes[1].plot(rec, prec, color=COULEURS[type_modele], lw=2, label=type_modele)
axes[0].plot([0, 1], [0, 1], ls="--", color=GRIS, lw=1, label="hasard")
axes[1].axhline(y_test.mean(), ls="--", color=GRIS, lw=1, label="hasard")
axes[0].set(title="Courbe ROC", xlabel="taux de faux positifs", ylabel="taux de vrais positifs (recall)")
axes[1].set(title="Courbe précision-recall", xlabel="recall", ylabel="précision")
for ax in axes:
    ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(IMG / "modele_courbes.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Matrice de confusion du modèle XGBoost au seuil 0,5
tn, fp, fn, tp = confusion_matrix(y_test, modeles["xgboost"].predict(X_test)).ravel()
pd.DataFrame([[tn, fp], [fn, tp]],
             index=["réellement resté", "réellement parti"],
             columns=["prédit resté", "prédit parti"])

## 4. Pourquoi les clients partent : explicabilité globale (SHAP)

Pour chaque client, **SHAP** mesure la contribution de chaque variable à sa prédiction :
positive = pousse vers le départ, négative = retient le client.
La moyenne des valeurs absolues donne l'**importance globale** de chaque variable.

In [ ]:
modele = modeles["xgboost"]
shap_test = modele.expliquer(X_test)

importance = shap_test.abs().mean().sort_values().tail(12)
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(importance.index, importance.values, color=BLEU, height=0.6)
for i, v in enumerate(importance.values):
    ax.text(v + 0.01, i, f"{v:.2f}", va="center", fontsize=9, color=GRIS)
ax.set(title="Importance globale des variables (moyenne |SHAP|, XGBoost)", xlabel="impact moyen sur la prédiction")
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(IMG / "shap_importance.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Vue détaillée : chaque point est un client ; la couleur = valeur de la variable (rouge = élevée)
shap.summary_plot(shap_test.values, X_test, max_display=10, show=False)
plt.tight_layout()
plt.show()

## 5. Expliquer un client en particulier

In [ ]:
proba_test = pd.Series(modele.predict_proba(X_test), index=X_test.index)
client_id = proba_test.idxmax()  # le client jugé le plus à risque
explication = modele.expliquer_client(X_test.loc[[client_id]], top=3)

print(f"Client {df.loc[client_id, 'customer_id']} : risque de départ {explication['proba_churn']:.0%}")
print("Principaux facteurs de risque :")
for variable, contribution in explication["facteurs"]:
    print(f"  - {variable} = {X_test.loc[client_id, variable]}  (contribution SHAP +{contribution})")

C'est ce qu'un conseiller rétention peut utiliser : pas seulement *« ce client va partir »*, mais *« il va partir
**parce que** … »*, donc quelle offre lui proposer (engagement, baisse de prix, support…).

## 6. Le pont avec les avis clients

On regroupe l'importance SHAP par **motif** (ex. `contrat_*` → résiliation, `internet_*` → réseau,
`paiement_*` / `montant_mensuel` → facturation, `support_technique` → service client), puis on la compare
à la **part des plaintes** de chaque motif dans les vrais avis négatifs (table dbt `mart_motifs_operateur`).

On compare les 4 motifs entre eux (hors « autre »), en % du total de ces 4 motifs.

In [ ]:
MOTIFS = ["reseau", "facturation", "resiliation", "service_client"]

poids_churn = importance_par_motif(shap_test).reindex(MOTIFS)
poids_churn = 100 * poids_churn / poids_churn.sum()

with duckdb.connect(str(DB), read_only=True) as con:
    plaintes = con.sql("""
        SELECT motif, sum(n_negatifs) AS n
        FROM mart_motifs_operateur
        GROUP BY motif
    """).df().set_index("motif")["n"].reindex(MOTIFS)
part_plaintes = 100 * plaintes / plaintes.sum()

pont = pd.DataFrame({"poids dans le churn (SHAP)": poids_churn, "part des plaintes (avis)": part_plaintes}).astype(float).round(1)
pont

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x = range(len(MOTIFS))
largeur = 0.38
barres1 = ax.bar([i - largeur / 2 for i in x], pont.iloc[:, 0], largeur, color=BLEU, label=pont.columns[0])
barres2 = ax.bar([i + largeur / 2 for i in x], pont.iloc[:, 1], largeur, color=ORANGE, label=pont.columns[1])
for barres in (barres1, barres2):
    for b in barres:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1, f"{b.get_height():.0f} %",
                ha="center", fontsize=9, color=GRIS)
ax.set_xticks(list(x), ["réseau", "facturation", "résiliation", "service client"])
ax.set(title="Ce qui fait partir (modèle) vs ce dont on se plaint (avis)", ylabel="% des 4 motifs")
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0f} %")
ax.legend(frameon=False)
ax.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig(IMG / "pont_motifs.png", dpi=120, bbox_inches="tight")
plt.show()

## 7. Sauvegarder le modèle

In [ ]:
chemin = Path("../data/modele_churn.joblib")  # data/ n'est pas versionné (.gitignore)
modele.save(chemin)
recharge = ModeleChurn.load(chemin)
print("Modèle sauvegardé et rechargé :", recharge.evaluer(X_test, y_test))

## 8. Synthèse

**Performance.** Les deux modèles atteignent une ROC-AUC d'environ **0,84**, stable en validation croisée
(écart-type ≈ 0,01), et détectent environ **3 partants sur 4** (recall ≈ 0,77) au seuil 0,5.
**XGBoost ne fait pas mieux que la régression logistique** : sur ce dataset, les relations sont assez simples
(quelques variables dominantes). En production, on pourrait donc choisir le modèle le plus simple ;
on garde XGBoost ici pour l'explicabilité SHAP par client, qui reste valable pour les deux.

**Pourquoi les clients partent.** Par ordre d'importance : le **contrat mensuel** (sans engagement),
une **faible ancienneté**, un **montant mensuel élevé**, la **fibre**, le **paiement par chèque électronique**.
Un contrat de 2 ans, au contraire, retient fortement.

**Le pont avec les avis.** Le motif **résiliation** (via le contrat) porte le plus gros du signal de churn,
alors qu'il est quasi absent des plaintes ; à l'inverse, le **service client**, très présent dans les avis
(surtout chez Orange), pèse peu dans le modèle. **Ce dont les clients se plaignent n'est pas ce qui les fait partir :**
les avis mesurent l'insatisfaction, le contrat mesure la liberté de partir. Les deux sources sont complémentaires.

**Limites.** Dataset Kaggle (clients américains, instantané sans dimension temporelle) ;
lien avec les avis au niveau des tendances seulement ; le rattachement variable → motif est un choix d'analyse.

**Pistes.** Ajuster le seuil de décision selon le coût d'une offre de rétention ; calibrer les probabilités ;
suivre la dérive du modèle dans le temps.